<a href="https://colab.research.google.com/github/pavithrasmv/Mchine-learning/blob/main/NLTK_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#import dataset
import pandas as pd

data = {
    "review_id": range(1, 31),
    "review": [
        "The phone arrived quickly and the packaging was excellent.",
        "The product quality is amazing and the battery lasts all day.",
        "I am very disappointed because the screen stopped working.",
        "Customer service solved my problem quickly and politely.",
        "The delivery was late and the package was damaged.",
        "Payment failed several times while placing my order.",
        "The headphones have excellent sound quality.",
        "The charger stopped working after two days.",
        "The delivery agent was friendly and arrived on time.",
        "I received the wrong product and need a replacement.",
        "The refund process was fast and easy.",
        "The app showed payment successful but the order was not created.",
        "The laptop performance is excellent for office work.",
        "Customer support did not respond to my complaint.",
        "The package arrived damaged and the product was broken.",
        "The product is good but delivery took too long.",
        "I love the camera quality and battery performance.",
        "The payment page keeps showing an error.",
        "The replacement was delivered quickly.",
        "The product stopped working within one week.",
        "The support team was helpful and professional.",
        "The order arrived earlier than expected.",
        "The refund has not arrived even after many days.",
        "The product packaging was poor and the box was damaged.",
        "The phone works perfectly and I am happy with the purchase.",
        "I could not complete payment using my card.",
        "The customer service team ignored my messages.",
        "The delivery was excellent and the package was safe.",
        "The device quality is poor and performance is slow.",
        "The replacement process was smooth and quick."
    ],
    "category": [
        "Delivery","Product Quality","Product Quality","Customer Service","Delivery",
        "Payment","Product Quality","Product Quality","Delivery","Product Quality",
        "Payment","Payment","Product Quality","Customer Service","Delivery",
        "Delivery","Product Quality","Payment","Delivery","Product Quality",
        "Customer Service","Delivery","Payment","Delivery","Product Quality",
        "Payment","Customer Service","Delivery","Product Quality","Customer Service"
    ]
}

df = pd.DataFrame(data)
df.head()
print("="*60)

#import libraries
!pip install -q nltk scikit-learn seaborn
import nltk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

 #Download NLTK Resources->natural language toolkit (Tokenization,Stop-word removal,Stemming,Lemmatization,Part-of-speech tagging etc)
 nltk.download('punkt')
nltk.download('punkt_tab')#: punkt_tab is required by current tokenization workflows
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')
nltk.download('vader_lexicon')

#Explore the Dataset
print(df.shape) #-tot number
print(df['category'].value_counts())
print(df.head())

print("="*60)
# Sentence and Word Tokenization
sample = df.loc[0, 'review']
sentences = nltk.sent_tokenize(sample) #Sentence tokenization divides a review into sentences
words = nltk.word_tokenize(sample) #Word tokenization divides a sentence into individual tokens.
print("Sentences:", sentences)
print("Words:", words)
print("="*60)

#Stopword Removal -> 'the', 'is', 'and', and 'to'
stop_words = set(stopwords.words('english'))
filtered_words = [
    word for word in words
    if word.lower() not in stop_words
]
print((filtered_words) )
print("="*60)

#Stemming ->trim word ends with ing,s using rules
stemmer = PorterStemmer()
stemmed_words = [stemmer.stem(word) for word in filtered_words]
print(stemmed_words)

#Lemmatization -> produce a meaningful dictionary form(better->good)
lemmatizer = WordNetLemmatizer()
lemmatized_words = [
    lemmatizer.lemmatize(word)
    for word in filtered_words
]
print(lemmatized_words)

#POS Tagging ->identifying the grammatical role of each word in a sentence( noun, verb, adjective, and adverb)
pos_tags = nltk.pos_tag(words)
for word, tag in pos_tags:
  print(word, "->", tag)

#Named Entity Recognition (NER) ->find important real-world entities in text and classify them into categories( persons, organizations,
#and geopolitical entities)
sample_ner = "Satya visited FITA Academy in Coimbatore."

ner_words = nltk.word_tokenize(sample_ner)
ner_pos = nltk.pos_tag(ner_words)
entities = nltk.ne_chunk(ner_pos)

print(entities)

# Text Normalization Function ->cleaning and standardizing text before giving it to an NLP model(upper,lower,remove punctuation,extra spaces etc)
def preprocess_text(text):
    words = nltk.word_tokenize(text.lower())

    words = [
        word for word in words
        if word.isalpha() and word not in stop_words
    ]

    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    return " ".join(words)

df['clean_text'] = df['review'].apply(preprocess_text)

df[['review', 'clean_text']].head()


#Sentiment Analysis using VADER ->Valence Aware Dictionary and sEntiment Reasoner(Positive, Negative, or Neutral sentiment).
sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    score = sia.polarity_scores(text)['compound']

    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

df['sentiment'] = df['review'].apply(get_sentiment)

df[['review', 'sentiment']].head(10)


#Explore Sentiment ->emotion or opinion expressed by the person.
print(df['sentiment'].value_counts())
sns.countplot(data=df, x='sentiment')
plt.title('Customer Sentiment Distribution')
plt.show()
print("="*60)


#TF-IDF Feature Extraction ->Term Frequency–Inverse Document Frequency. converts text into numerical vectors.
X = df['clean_text']
y = df['category']
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.25, random_state=42, stratify=y )
tfidf = TfidfVectorizer( max_features=1000, ngram_range=(1, 2) )
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
print("Training matrix:", X_train_tfidf.shape)
print("Testing matrix:", X_test_tfidf.shape)

#Train the Text Classification Model
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, zero_division=0))

#Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
plt.figure(figsize=(8, 5))
sns.heatmap( cm, annot=True, fmt='d', xticklabels=model.classes_, yticklabels=model.classes_ )
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

#Build a Complete Prediction Function
def predict_review(review):
    clean = preprocess_text(review)

    sentiment_score = sia.polarity_scores(review)['compound']

    if sentiment_score >= 0.05:
        sentiment = "Positive"
    elif sentiment_score <= -0.05:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"

    vector = tfidf.transform([clean])
    category = model.predict(vector)[0]

    return category, sentiment

new_review = "The delivery was very late and the package was damaged."

category, sentiment = predict_review(new_review)

print("Review:", new_review)
print("Predicted Category:", category)
print("Sentiment:", sentiment)


#Run Multiple New Reviews
new_reviews = [
    "The phone battery is excellent and the product quality is amazing.",
    "My payment failed and I could not place the order.",
    "The customer support team never replied to me.",
    "The package arrived late and was damaged."
]

for review in new_reviews:
    category, sentiment = predict_review(review)
    print("Review:", review)
    print("Category:", category)
    print("Sentiment:", sentiment)
    print("-" * 70)

# Business-Level Analysis
summary = (
df.groupby(['category', 'sentiment']) .size() .reset_index(name='review_count') )
print(summary.sort_values('review_count', ascending=False))
